# Sales Incentive Compensation Simulator – Analysis Notebook

This notebook provides an end-to-end walkthrough of the Sales Incentive
Compensation Simulator, covering:

1. **Data Loading** – Load generated CSV files (or regenerate them)
2. **Exploratory Data Analysis** – Distributions, attainment, deal patterns
3. **Incentive Engine Walkthrough** – Step-by-step commission calculation
4. **What-if Simulations** – Model alternative plan scenarios
5. **Summary Statistics** – Key business metrics


In [ ]:
import os
import sys

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.config_loader import load_config
from src.data_generator import generate_all_data
from src.incentive_engine import run_incentive_engine, calculate_attainment, calculate_payouts
from src.simulator import simulate_incentives, compare_scenarios, get_scenario_summary

# Plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

print('Environment ready.')

---
## 1. Load Configuration & Data


In [ ]:
config = load_config()
print(f"Plan version : {config['version']}")
print(f"Effective from: {config['effective_from']}")
print(f"Roles         : {list(config['roles'].keys())}")
print(f"Regions       : {config['regions']}")

In [ ]:
# Generate or load data from the data/ directory
data_dir = os.path.join(project_root, 'data')
required_files = ['sales_reps.csv', 'sales_transactions.csv', 'incentive_plan.csv', 'calendar.csv']

if all(os.path.isfile(os.path.join(data_dir, f)) for f in required_files):
    print('Loading existing CSVs ...')
    datasets = {
        'sales_reps':         pd.read_csv(os.path.join(data_dir, 'sales_reps.csv')),
        'sales_transactions': pd.read_csv(os.path.join(data_dir, 'sales_transactions.csv')),
        'incentive_plan':     pd.read_csv(os.path.join(data_dir, 'incentive_plan.csv')),
        'calendar':           pd.read_csv(os.path.join(data_dir, 'calendar.csv')),
    }
else:
    print('Generating synthetic data ...')
    datasets = generate_all_data(output_dir=data_dir, seed=42)

reps_df  = datasets['sales_reps']
sales_df = datasets['sales_transactions']

for name, df in datasets.items():
    print(f'  {name:<20} {len(df):>6,} rows  |  cols: {list(df.columns)}')

---
## 2. Exploratory Data Analysis


In [ ]:
# --- 2.1  Sales Reps: distribution by role and region ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

role_counts = reps_df['role'].value_counts()
axes[0].barh(role_counts.index, role_counts.values, color=sns.color_palette('muted', len(role_counts)))
axes[0].set_title('Rep Count by Role')
axes[0].set_xlabel('Number of Reps')
for i, v in enumerate(role_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center')

region_counts = reps_df['region'].value_counts()
axes[1].pie(region_counts.values, labels=region_counts.index,
            autopct='%1.1f%%', colors=sns.color_palette('pastel', len(region_counts)))
axes[1].set_title('Rep Distribution by Region')

plt.tight_layout()
plt.show()

In [ ]:
# --- 2.2  Quota distribution by role ---
fig, ax = plt.subplots(figsize=(13, 5))
role_order = reps_df.groupby('role')['quota'].median().sort_values(ascending=False).index
sns.boxplot(data=reps_df, x='role', y='quota', order=role_order,
            palette='muted', ax=ax)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Quota Distribution by Role')
ax.set_xlabel('')
ax.set_ylabel('Annual Quota (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.3  Deal amount distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all deal amounts (log scale x-axis)
axes[0].hist(sales_df['deal_amount'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xscale('log')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_title('Deal Amount Distribution (log scale)')
axes[0].set_xlabel('Deal Amount')
axes[0].set_ylabel('Count')

# Median deal size by role
merged = sales_df.merge(reps_df[['rep_id', 'role']], on='rep_id')
role_deal_med = merged.groupby('role')['deal_amount'].median().sort_values(ascending=False)
axes[1].barh(role_deal_med.index, role_deal_med.values / 1_000,
             color=sns.color_palette('muted', len(role_deal_med)))
axes[1].set_title('Median Deal Size by Role')
axes[1].set_xlabel('Median Deal ($K)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
# --- 2.4  Monthly revenue trend ---
sales_df['deal_date'] = pd.to_datetime(sales_df['deal_date'])
monthly = sales_df.resample('ME', on='deal_date')['deal_amount'].sum().reset_index()
monthly.columns = ['month', 'revenue']

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(monthly['month'], monthly['revenue'] / 1e6, color='steelblue', alpha=0.8)
ax.plot(monthly['month'], monthly['revenue'] / 1e6, 'o-', color='navy', lw=1.5)
ax.set_title('Monthly Revenue – 2024')
ax.set_ylabel('Revenue ($M)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.5  Revenue by product category and customer segment ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prod_rev = sales_df.groupby('product_category')['deal_amount'].sum().sort_values(ascending=True)
axes[0].barh(prod_rev.index, prod_rev.values / 1e6, color=sns.color_palette('muted', len(prod_rev)))
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
axes[0].set_title('Revenue by Product Category')
axes[0].set_xlabel('Total Revenue')

seg_rev = sales_df.groupby('customer_segment')['deal_amount'].sum().sort_values(ascending=False)
axes[1].pie(seg_rev.values, labels=seg_rev.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(seg_rev)))
axes[1].set_title('Revenue Share by Customer Segment')

plt.tight_layout()
plt.show()

---
## 3. Incentive Engine Walkthrough


In [ ]:
# Run the full incentive engine
results = run_incentive_engine(sales_df, reps_df, config)
print(f'Results shape: {results.shape}')
results.head()

In [ ]:
# --- 3.1  Attainment distribution ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(results['attainment_pct'] * 100, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(100, color='red', linestyle='--', linewidth=1.5, label='Quota (100%)')
ax.axvline(results['attainment_pct'].mean() * 100, color='orange',
           linestyle='--', linewidth=1.5, label=f'Mean ({results["attainment_pct"].mean()*100:.1f}%)')
ax.set_title('Rep Attainment Distribution')
ax.set_xlabel('Attainment %')
ax.set_ylabel('Number of Reps')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Reps above quota : {(results['attainment_pct'] >= 1.0).sum()} / {len(results)} "
      f"({(results['attainment_pct'] >= 1.0).mean()*100:.1f}%)")

In [ ]:
# --- 3.2  Payout components ---
fig, ax = plt.subplots(figsize=(12, 5))

plot_df = results.sort_values('attainment_pct')
x = range(len(plot_df))
ax.bar(x, plot_df['base_commission'] / 1_000, label='Base Commission', color='steelblue', alpha=0.8)
ax.bar(x, plot_df['accelerator_bonus'] / 1_000,
       bottom=plot_df['base_commission'] / 1_000,
       label='Accelerator Bonus', color='orange', alpha=0.8)
ax.set_title('Payout Composition per Rep (sorted by attainment)')
ax.set_xlabel('Reps (sorted by attainment, low → high)')
ax.set_ylabel('Payout ($K)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.3  Attainment vs. Payout scatter ---
fig, ax = plt.subplots(figsize=(10, 6))
role_palette = dict(zip(results['role'].unique(), sns.color_palette('tab10', results['role'].nunique())))
for role, grp in results.groupby('role'):
    ax.scatter(grp['attainment_pct'] * 100, grp['total_payout'] / 1_000,
               label=role, alpha=0.7, s=60, color=role_palette[role])
ax.axvline(100, color='red', linestyle='--', linewidth=1, label='Quota Line')
ax.set_title('Attainment % vs. Total Payout')
ax.set_xlabel('Attainment %')
ax.set_ylabel('Total Payout ($K)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.4  Payout summary by region ---
region_summary = (
    results.groupby('region')
    .agg(
        reps=('rep_id', 'count'),
        total_revenue=('total_sales', 'sum'),
        total_payout=('total_payout', 'sum'),
        avg_attainment=('attainment_pct', 'mean'),
        pct_above_quota=('attainment_pct', lambda x: (x >= 1.0).mean())
    )
    .reset_index()
)
region_summary['payout_ratio'] = region_summary['total_payout'] / region_summary['total_revenue']
region_summary['avg_attainment'] = (region_summary['avg_attainment'] * 100).round(1)
region_summary['pct_above_quota'] = (region_summary['pct_above_quota'] * 100).round(1)
region_summary['payout_ratio'] = (region_summary['payout_ratio'] * 100).round(2)

display_cols = ['region', 'reps', 'total_revenue', 'total_payout',
                'avg_attainment', 'pct_above_quota', 'payout_ratio']
print(region_summary[display_cols].to_string(index=False))

---
## 4. What-if Simulation Examples


In [ ]:
# --- 4.1  Scenario: raise quotas by 10% ---
scenario_a = simulate_incentives(sales_df, reps_df, params={'quota_adjustment_pct': 0.10})
summary_base   = get_scenario_summary(simulate_incentives(sales_df, reps_df, params={}))
summary_scen_a = get_scenario_summary(scenario_a)

print('Scenario A: +10% Quota')
print(f"  Base payout    : ${summary_base['total_payout']:>12,.0f}")
print(f"  Scenario payout: ${summary_scen_a['total_payout']:>12,.0f}")
print(f"  Delta          : ${summary_scen_a['total_payout'] - summary_base['total_payout']:>+12,.0f}")
print(f"  Base avg attain: {summary_base['avg_attainment']*100:.1f}%")
print(f"  Scen avg attain: {summary_scen_a['avg_attainment']*100:.1f}%")

In [ ]:
# --- 4.2  Scenario comparison: higher accelerator rate ---
comparison = compare_scenarios(
    base_params={},
    scenario_params={'accelerator_rate': 0.25},
    sales_df=sales_df,
    reps_df=reps_df,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of payout delta
axes[0].hist(comparison['payout_delta'] / 1_000, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Payout Delta: Higher Accelerator (25% vs 15%)')
axes[0].set_xlabel('Payout Delta ($K)')
axes[0].set_ylabel('Number of Reps')

# Scatter: base payout vs delta
above = comparison[comparison['payout_delta'] > 0]
equal = comparison[comparison['payout_delta'] == 0]
axes[1].scatter(above['total_payout_base'] / 1_000, above['payout_delta'] / 1_000,
                alpha=0.7, label='Benefiting reps', color='green', s=50)
axes[1].scatter(equal['total_payout_base'] / 1_000, equal['payout_delta'] / 1_000,
                alpha=0.5, label='No change', color='grey', s=50)
axes[1].set_title('Base Payout vs. Delta')
axes[1].set_xlabel('Base Payout ($K)')
axes[1].set_ylabel('Payout Delta ($K)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Reps benefiting from higher accelerator: {(comparison['payout_delta'] > 0).sum()}")
print(f"Total extra cost                       : ${comparison['payout_delta'].sum():,.0f}")

In [ ]:
# --- 4.3  Multi-scenario comparison chart ---
scenarios = {
    'Base':                    {},
    'Quota +10%':              {'quota_adjustment_pct': 0.10},
    'Quota -10%':              {'quota_adjustment_pct': -0.10},
    'Accel 20%':               {'accelerator_rate': 0.20},
    'Accel 10%':               {'accelerator_rate': 0.10},
    'Enterprise AE only':      {'role_filter': ['Enterprise AE']},
    'North America only':      {'region_filter': ['North America']},
}

scenario_results = {}
for name, params in scenarios.items():
    sim = simulate_incentives(sales_df, reps_df, params=params)
    scenario_results[name] = get_scenario_summary(sim)

summary_df = pd.DataFrame(scenario_results).T.reset_index()
summary_df.columns = ['scenario'] + list(summary_df.columns[1:])
summary_df['total_payout_M'] = summary_df['total_payout'] / 1e6
summary_df['avg_attainment_pct'] = summary_df['avg_attainment'] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['steelblue' if s == 'Base' else 'lightcoral' for s in summary_df['scenario']]
axes[0].barh(summary_df['scenario'], summary_df['total_payout_M'], color=colors)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
axes[0].set_title('Total Payout by Scenario')
axes[0].set_xlabel('Total Payout')

axes[1].barh(summary_df['scenario'], summary_df['avg_attainment_pct'], color=colors)
axes[1].axvline(100, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Average Attainment by Scenario')
axes[1].set_xlabel('Avg Attainment %')

plt.tight_layout()
plt.show()

print(summary_df[['scenario', 'total_payout_M', 'avg_attainment_pct', 'pct_above_quota', 'payout_ratio']].to_string(index=False))

---
## 5. Summary Statistics


In [ ]:
base_summary = get_scenario_summary(results)

print('=' * 55)
print(f"  {'Sales Incentive Simulator – Key Metrics':^51}")
print('=' * 55)
print(f"  {'Total Revenue':<35} ${base_summary['total_revenue']:>12,.0f}")
print(f"  {'Total Payout':<35} ${base_summary['total_payout']:>12,.0f}")
print(f"  {'Payout Ratio':<35} {base_summary['payout_ratio']*100:>11.2f}%")
print(f"  {'Avg Attainment':<35} {base_summary['avg_attainment']*100:>11.1f}%")
print(f"  {'% Reps Above Quota':<35} {base_summary['pct_above_quota']*100:>11.1f}%")
print(f"  {'Total Transactions':<35} {len(sales_df):>12,}")
print(f"  {'Total Reps':<35} {len(reps_df):>12,}")
print(f"  {'Avg Deal Size':<35} ${sales_df['deal_amount'].mean():>12,.0f}")
print(f"  {'Median Deal Size':<35} ${sales_df['deal_amount'].median():>12,.0f}")
print('=' * 55)

In [ ]:
# Heatmap: avg attainment by region × role
pivot = results.pivot_table(
    values='attainment_pct', index='region', columns='role',
    aggfunc='mean'
) * 100

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='RdYlGn',
    center=100, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Attainment %'}
)
ax.set_title('Average Attainment % by Region × Role')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()